In [1]:
import pandas as pd
import os

In [16]:
base_dir = "C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/강우량"

# 3개년 여름철 데이터 통합 (2023, 2024, 2025년 6, 7, 8월)
years = [2023, 2024, 2025]
months = ['06', '07', '08']

df_list = []

for y in years:
    for m in months:
        file_name = f'서울시_강우량_정보_{y}년{m}월.csv'
        file_path = f"{base_dir}/{file_name}"
        
        # 파일이 존재하는 경우에만 읽어오기
        if os.path.exists(file_path):
            try:
                temp_df = pd.read_csv(file_path, encoding='cp949')
            except UnicodeDecodeError:
                temp_df = pd.read_csv(file_path, encoding='utf-8')
            
            df_list.append(temp_df)
        else:
            print(f"경고: {file_name} 파일을 찾을 수 없습니다.")

In [17]:
# 모든 데이터프레임을 하나로 병합
rain_df = pd.concat(df_list, ignore_index=True)
print(f"총 데이터 건수: {len(rain_df):,}건")

총 데이터 건수: 1,883,955건


In [18]:
# 2. 시계열 데이터 전처리
# '자료수집 시각'에 분 단위와 초 단위 형식이 섞여 있으므로 format='mixed' 적용
# 변환할 수 없는 이상한 문자열은 강제로 결측치(NaT)로 만든 후 제거
rain_df['자료수집 시각'] = pd.to_datetime(rain_df['자료수집 시각'], format='mixed', errors='coerce')
rain_df = rain_df.dropna(subset=['자료수집 시각'])

# 결측치나 이상치(예: 강우량이 음수인 경우 등) 제거
rain_df = rain_df[rain_df['10분우량'] >= 0]

In [19]:
# 인덱스가 '자료수집 시각'이 아니라면 시간 인덱스로 설정
if rain_df.index.name != '자료수집 시각':
    rain_df.set_index('자료수집 시각', inplace=True)

In [20]:
# 3. 관측소별 시간당/일일 강수량 리샘플링
hourly_rain = rain_df.groupby(['구청명', '강우량계명'])['10분우량'].resample('1h').sum().reset_index()
daily_rain = rain_df.groupby(['구청명', '강우량계명'])['10분우량'].resample('1D').sum().reset_index()

# 리샘플링된 결과에서 '연도' 컬럼 생성
hourly_rain['연도'] = hourly_rain['자료수집 시각'].dt.year
daily_rain['연도'] = daily_rain['자료수집 시각'].dt.year

In [21]:
# 4. 연도별 상위 30개 기록의 평균 추출 (매년 가장 비가 많이 온 30시간 / 30일의 평균)
# nlargest(30)을 통해 강수량 기준 상위 30개 값을 가져온 뒤 평균 산출
yearly_top30_hourly = hourly_rain.groupby(['구청명', '강우량계명', '연도'])['10분우량'].apply(lambda x: x.nlargest(30).mean()).reset_index(name='연도별_상위30_평균_시우량')
yearly_top30_daily = daily_rain.groupby(['구청명', '강우량계명', '연도'])['10분우량'].apply(lambda x: x.nlargest(30).mean()).reset_index(name='연도별_상위30_평균_일강수량')

In [ ]:
# 5. 3개년 통합 평균치 산출 (3년 동안의 꾸준한 취약성 확인)
avg_top30_hourly = yearly_top30_hourly.groupby(['구청명', '강우량계명'])['연도별_상위30_평균_시우량'].mean().reset_index(name='최종_상위30_평균_시우량')
avg_top30_daily = yearly_top30_daily.groupby(['구청명', '강우량계명'])['연도별_상위30_평균_일강수량'].mean().reset_index(name='최종_상위30_평균_일강수량')

In [23]:
# 두 지표 병합
gauge_features = pd.merge(avg_top30_hourly, avg_top30_daily, on=['구청명', '강우량계명'])

In [24]:
# 6. 자치구별 노출 지표 집계
# 동일 자치구 내 여러 관측소 중 가장 취약한(수치가 높은) 관측소 기준 채택
gu_exposure = gauge_features.groupby('구청명')[['최종_상위30_평균_시우량', '최종_상위30_평균_일강수량']].max().reset_index()
gu_exposure.rename(columns={'구청명': '자치구'}, inplace=True)

In [25]:
# 7. 커스텀 정방향 정규화 (최솟값 0.1 보정)
def custom_min_max_scale(series, a=0.1, b=1.0):
    s_min = series.min()
    s_max = series.max()
    return a + ((series - s_min) * (b - a)) / (s_max - s_min)

gu_exposure['E_최대시우량_정규화'] = custom_min_max_scale(gu_exposure['최종_상위30_평균_시우량'])
gu_exposure['E_최대일강수량_정규화'] = custom_min_max_scale(gu_exposure['최종_상위30_평균_일강수량'])

In [26]:
# 최종 노출 지수 (E) 산출
gu_exposure['E_최종지수'] = (gu_exposure['E_최대시우량_정규화'] + gu_exposure['E_최대일강수량_정규화']) / 2

In [28]:
# 결과 확인
gu_exposure_sorted = gu_exposure.sort_values(by='E_최종지수', ascending=False).reset_index(drop=True)
print("\n[자치구별 상위 30일/30시간 평균 기후 노출(E) 지수 산출 결과]")
display(gu_exposure_sorted)


[자치구별 상위 30일/30시간 평균 기후 노출(E) 지수 산출 결과]


,자치구,최종_상위30_평균_시우량,최종_상위30_평균_일강수량,E_최대시우량_정규화,E_최대일강수량_정규화,E_최종지수
0,구로구,19.005556,28.027778,1.000000,0.991429,0.995714
1,도봉구,15.905556,28.083333,0.548381,1.000000,0.774191
2,강서구,16.216667,26.500000,0.593705,0.755714,0.674710
3,성북구,15.172222,27.133333,0.441547,0.853429,0.647488
4,은평구,16.377778,25.950000,0.617176,0.670857,0.644017
5,노원구,15.438889,26.794444,0.480396,0.801143,0.640769
6,중랑구,15.555556,26.494444,0.497392,0.754857,0.626125
7,동대문구,15.316667,26.550000,0.462590,0.763429,0.613009
8,강북구,15.500000,26.355556,0.489299,0.733429,0.611364
9,송파구,15.827778,25.655556,0.537050,0.625429,0.581239


### 1. 집중호우 지표 산출
- 10분 단위 우량 -> 홍수에 직접 타격을 주는 극한 강수 지표로 리샘플링
    - 단일 이상치로 인한 과대적합을 방지하고, 여름철 내내 지속되는 실질적 호우 타격을 반영하기 위해 상위 30개의 기록을 평균 내어 사용
    - 일 최대 강수량 : 매년 비가 가장 많이 내린 상위 30일의 강수량을 평균 내어 3년치 통합 도출
    - 시우량 : 매년 비가 가장 많이 내린 상위 30시간의 강수량을 평균 내어 3년치 통합 도출

### 2. 위치 기반 매핑 및 자치구별 집계
- 민감도 지표와 기준을 맞추기 위해 자치구 단위로 데이터 병합
    - 데이터에 포함되어 있는 구청명 기준으로 groupby
    - 해당 자치구 내 측정소들의 평균 또는 최댓값 (동일 구 안에 관측소 여러개)

### 3. 취약성 평가 지수
- 일 최대 강수량, 시간 최대 강수량을 정규화 후 평균 : 최종 E
    - 홍수 취약성 공식에서 노출 지표(Exposure, E)
    - 홍수 취약성 공식(V = E * S - AC)의 곱셈 연산 특성상 특정 지표가 0이 될 경우 전체 위험도가 상쇄되는 오류(Zero-Multiplier)를 방지하기 위해 최솟값을 0.1로 보정하는 정방향 스케일링(0.1 ~ 1.0) 적용

In [37]:
# 1. 2023-2024년 데이터 로드 및 전처리
df_2324 = pd.read_csv('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/토지/빗물_등이_지하로_스며들수_없게_하는_불투수면_현황_20260527135824.csv')
df_2324 = df_2324.iloc[2:].copy()

# '서울특별시' 데이터만 추출하고, 자치구가 아닌 '소계'는 제외
df_seoul = df_2324[(df_2324['구분(1)'] == '서울특별시') & (df_2324['구분(2)'] != '소계')].copy()

# 필요한 컬럼(자치구명, 23년 비율, 24년 비율) 추출 및 이름 변경
imperv_2324 = df_seoul[['구분(2)', '2023.1', '2024.1']].copy()
imperv_2324.columns = ['자치구', '2023', '2024']

# 비율 데이터를 숫자(실수)형으로 변환
imperv_2324['2023'] = imperv_2324['2023'].astype(float)
imperv_2324['2024'] = imperv_2324['2024'].astype(float)

In [39]:
# 2. 2025년 데이터 로드 및 전처리 (이전 업로드 파일 활용)
df_25 = pd.read_excel('C:/Users/hyeon/OneDrive/Desktop/2026-1/공모전/데이터분석/데이터/토지/불투수면적_현황자료_2025.xlsx')
imperv_25 = df_25[['자치구', '불투수면적 비율(%)']].copy()
imperv_25.columns = ['자치구', '2025']

In [41]:
# 3. 2023~2025년 데이터 자치구 기준으로 가로 병합
imperv_merged = pd.merge(imperv_2324, imperv_25, on='자치구', how='left')

In [43]:
# 4. 연도별 형태로 변환
imperv_yearly = imperv_merged.melt(id_vars=['자치구'], var_name='연도', value_name='불투수면적 비율(%)')
imperv_yearly['연도'] = imperv_yearly['연도'].astype(int)

In [45]:
display(imperv_yearly)

,자치구,연도,불투수면적 비율(%)
0,종로구,2023,41.19
1,중구,2023,76.25
2,용산구,2023,47.78
3,성동구,2023,62.82
4,광진구,2023,63.33
...,...,...,...
70,관악구,2025,39.52
71,서초구,2025,36.76
72,강남구,2025,55.44
73,송파구,2025,57.87


In [46]:
# 5. 연도별 강수량 지표 병합 
yearly_gauge_features = pd.merge(yearly_top30_hourly, yearly_top30_daily, on=['구청명', '강우량계명', '연도'])

In [47]:
# 6. 강수량 데이터와 3개년 불투수면적 데이터(imperv_yearly) 병합
yearly_merged = pd.merge(yearly_gauge_features, imperv_yearly, left_on=['구청명', '연도'], right_on=['자치구', '연도'], how='inner')

In [ ]:
# 7. 매년 달라진 불투수면적을 반영해 유효 강수량(표면유출량) 계산
yearly_merged['연도별_유효_시우량'] = yearly_merged['연도별_상위30_평균_시우량'] * (yearly_merged['불투수면적 비율(%)'] / 100)
yearly_merged['연도별_유효_일강수량'] = yearly_merged['연도별_상위30_평균_일강수량'] * (yearly_merged['불투수면적 비율(%)'] / 100)

In [49]:
# 8. 3개년 평균 유효 강수량 산출
avg_eff_gauge = yearly_merged.groupby(['자치구', '강우량계명'])[['연도별_유효_시우량', '연도별_유효_일강수량']].mean().reset_index()
avg_eff_gauge.rename(columns={'연도별_유효_시우량': '최종_평균_유효_시우량', '연도별_유효_일강수량': '최종_평균_유효_일강수량'}, inplace=True)

In [ ]:
# 9. 자치구별 대푯값 집계 (동일 자치구 내 가장 위험한 관측소 기준 최댓값 채택)
gu_runoff = avg_eff_gauge.groupby('자치구')[['최종_평균_유효_시우량', '최종_평균_유효_일강수량']].max().reset_index()

In [51]:
# 10. 표면유출 위험지수 커스텀 정규화 함수
def custom_min_max_scale(series, a=0.1, b=1.0):
    s_min = series.min()
    s_max = series.max()
    return a + ((series - s_min) * (b - a)) / (s_max - s_min)

gu_runoff['유효_최대시우량_정규화'] = custom_min_max_scale(gu_runoff['최종_평균_유효_시우량'])
gu_runoff['유효_최대일강수량_정규화'] = custom_min_max_scale(gu_runoff['최종_평균_유효_일강수량'])

In [52]:
# 11. 최종 표면유출 위험지수 산출
gu_runoff['표면유출_위험지수'] = (gu_runoff['유효_최대시우량_정규화'] + gu_runoff['유효_최대일강수량_정규화']) / 2

In [53]:
# 결과 확인
runoff_sorted = gu_runoff.sort_values(by='표면유출_위험지수', ascending=False).reset_index(drop=True)
print("\n[자치구별 3개년 동기화 표면유출 위험지수 산출 결과]")
display(runoff_sorted[['자치구', '최종_평균_유효_시우량', '최종_평균_유효_일강수량', '표면유출_위험지수']])


[자치구별 3개년 동기화 표면유출 위험지수 산출 결과]


,자치구,최종_평균_유효_시우량,최종_평균_유효_일강수량,표면유출_위험지수
0,동대문구,11.506557,19.968397,0.960891
1,구로구,12.116447,17.860347,0.915145
2,중구,10.778576,18.732163,0.864447
3,양천구,10.456429,17.557779,0.796517
4,중랑구,9.471676,16.143399,0.676437
5,송파구,9.681632,15.684416,0.671425
6,성동구,9.151549,15.678308,0.637187
7,금천구,9.164189,15.240473,0.620374
8,동작구,9.003622,15.436564,0.617971
9,영등포구,9.198551,15.024662,0.613890


### 1. 연도별 동기화 기반 표면유출량(유효 강수량) 추정
- 시간의 흐름에 따른 도심 지표면 변화를 반영하기 위해, 각 연도별 강수량 지표와 해당 연도(2023~2025년)의 불투수면적 비율을 1:1로 결합하여 실제 표면으로 흘러넘치는 빗물의 양을 정밀하게 산출
    - 연도별 유효 시우량
        - 해당 연도 상위 30시간 평균 시우량 * 해당 연도 불투수면적 비율
        - 단기간에 땅에 스며들지 못하는 빗물의 양을 매년 달라진 지표면 상태에 맞춰 추정
    - 연도별 유효 일강수량
        - 해당 연도 상위 30일 평균 일강수량 * 해당 연도 불투수면적 비율
        - 하루 동안 침수 취약 구역에 지속적으로 누적되는 빗물의 양을 매년 달라진 지표면 상태에 맞춰 추정

### 2. 위치 기반 데이터 결합 및 3개년 평균 대푯값 도출
- 기후 지표와 토지피복 지표를 결합한 후 통계적 안정성을 위해 통합 및 대표성 확보
    - 3개년 평균 산출: 연도별로 계산된 유효 시우량과 유효 일강수량을 관측소 단위로 3개년 평균 내어 특정 연도의 극단적 이상치 영향 최소화
    - 자치구 기준 병합: 관측소가 위치한 구청명을 자치구 기준으로 매핑
    - 최댓값 채택: 동일 자치구 내에 여러 관측소가 있는 경우, 해당 자치구의 가장 취약한(유효 강수량이 높은) 국지적 상태를 반영하기 위해 관측소 중 최댓값을 구의 대푯값으로 선정

### 3. 표면 유출 위험지수 산출
- 자치구별 대푯값으로 도출된 최종 유효 시우량과 유효 일강수량을 각각 정규화 후 평균
    - 커스텀 정방향 정규화 (0.1 ~ 1.0): 취약성 통합 연산 시 발생할 수 있는 0 곱셈(Zero-Multiplier) 상쇄 오류를 방지하기 위해, 최솟값을 0.1로 보정하는 스케일링 적용
    - 정규화된 두 유효 강수량 지표를 평균 내어 최종 표면유출 위험지수 도출